<a href="https://colab.research.google.com/github/sethkipsangmutuba/Database-Management-System/blob/main/Week_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5: Transaction Processing & Concurrency Control  

## 1. Introduction  
Modern Database Management Systems (DBMS) serve multiple users and applications concurrently. **Transactions** — sequences of operations performed as a single logical unit — are at the heart of database interaction.  
When many transactions execute simultaneously, they must appear as if they ran one after another (**serial execution**), even though they overlap in time. Achieving this requires **concurrency control** mechanisms that guarantee consistency while maximizing throughput.  

## 2. Objectives Recap  
By the end of this lecture, you should be able to:  
- Define and explain the **ACID** properties of transactions.  
- Describe the **lifecycle** of a transaction from start to commit or abort.  
- Differentiate between **serializability** and other isolation levels.  
- Explain and implement basic concurrency control methods such as **Two-Phase Locking (2PL)** and **Multiversion Concurrency Control (MVCC)**.  
- Identify and explain **deadlocks**, as well as strategies for detection and prevention.  
- Understand **snapshot isolation** in modern DBMS.  

## 3. ACID Properties  
Transactions must meet the **ACID** properties:  

### 3.1 Atomicity  
- **All or nothing**: Either all operations complete successfully, or none are applied.  
- Enforced via **undo logging** or rollback.  
Example:  
$$
T_1: \text{Withdraw } \$100 \ \text{from A} \quad\to\quad T_1: \text{Deposit } \$100 \ \text{to B}
$$  
If the deposit fails after the withdrawal, the withdrawal is undone.  

### 3.2 Consistency  
- Ensures the database transitions from one valid state to another.  
- Enforces **integrity constraints** (foreign keys, unique constraints, business rules).  

### 3.3 Isolation  
- Concurrent transactions must not interfere with each other’s intermediate states.  
- Controlled by **isolation levels** (Section 4).  

### 3.4 Durability  
- Once committed, effects persist even after a crash.  
- Achieved via **Write-Ahead Logging (WAL)**, backups, and replication.  

## 4. Transaction Lifecycle  
Stages:  
1. **Begin**: Transaction starts.  
2. **Read/Write**: Execute SQL operations.  
3. **Validation**: Check constraints.  
4. **Commit**: Make changes permanent.  
5. **Abort/Rollback**: Undo all changes if needed.  

Example Lifecycle Table:  

| Step   | Action                  | Possible Issues             |
|--------|------------------------|-----------------------------|
| Begin  | BEGIN TRANSACTION;     | None                        |
| Read   | Read product price     | Dirty reads                 |
| Write  | Update inventory       | Lost updates                |
| Commit | Make changes permanent | Crash before log flush      |
| Abort  | Roll back changes      | Cascading aborts            |  

## 5. Isolation & Serializability  

### 5.1 What is Serializability?  
- Highest isolation: concurrent execution equivalent to some serial order.  
- **Conflict serializability**: No conflicting operations are reordered.  
- **View serializability**: Equivalent results even with some reordering.  

### 5.2 Isolation Levels (ANSI SQL)  

| Level            | Prevents                          | Allows                                 |
|------------------|-----------------------------------|----------------------------------------|
| Read Uncommitted | None                              | Dirty, non-repeatable, phantom reads   |
| Read Committed   | Dirty reads                       | Non-repeatable, phantom reads          |
| Repeatable Read  | Dirty, non-repeatable reads       | Phantom reads                          |
| Serializable     | Dirty, non-repeatable, phantom    | None                                   |  

Example: At Read Committed, two transactions might see different values of the same row due to intermediate commits.  

## 6. Concurrency Control Techniques  

### 6.1 Two-Phase Locking (2PL)  
- Acquire all locks before releasing any.  
- **Growing phase**: Acquire locks.  
- **Shrinking phase**: Release locks.  

Lock types:  
- **Shared (S)**: Read-only.  
- **Exclusive (X)**: Write.  

Pros: Guarantees conflict serializability.  
Cons: Can cause deadlocks.  

Example Deadlock:  
$$
T_1: \text{Lock-S}(A) \ \to\ \text{Lock-X}(B)  
$$
$$
T_2: \text{Lock-S}(B) \ \to\ \text{Lock-X}(A)  
$$
Both wait forever.  

### 6.2 Deadlock Detection & Prevention  
- **Detection**: Wait-for graph; cycle = deadlock.  
- **Prevention**:  
  - Timeouts  
  - Resource ordering  
  - **Wound–Wait / Wait–Die** policies  

### 6.3 Multiversion Concurrency Control (MVCC)  
- Multiple versions of each row kept.  
- Readers see snapshot from transaction start.  
- Writers create new version without blocking readers.  

Pros: No reader–writer blocking; supports snapshot isolation.  
Cons: Higher storage use; version cleanup required.  

**Snapshot Isolation**:  
- Reads from a single consistent point in time.  
- Prevents dirty/non-repeatable reads.  
- May allow **write skew**.  

## 7. Join to Execution Models  
- Locking can stall pipeline execution.  
- MVCC fits iterator model — old versions can feed query plans without blocking.  

## 8. Practical Activities  

### 8.1 Lab Simulation  
- Build a lock table.  
- Simulate two transactions accessing the same resource.  
- Demonstrate 2PL and deadlock resolution.  

### 8.2 Reading  
- Kleppmann, *Designing Data-Intensive Applications*, Ch. 7–8.  
- PostgreSQL docs on MVCC and isolation levels.  

## 9. Summary Table  

| Concept             | Key Points                  | Pros                | Cons               |
|---------------------|-----------------------------|---------------------|--------------------|
| ACID                | Transaction reliability     | Strong guarantees   | Overhead           |
| 2PL                 | Lock-based, serializable    | Simple              | Deadlocks          |
| MVCC                | Version-based               | High concurrency    | Storage cost       |
| Deadlock Detection  | Wait-for graph               | Finds deadlocks     | Overhead           |
| Snapshot Isolation  | MVCC-based isolation        | No read locks       | Allows write skew  |  

## 10. Conclusion  
- **ACID** is the foundation of reliable transactions.  
- Concurrency control ensures isolation and prevents anomalies.  
- Locking and versioning are main strategies.  
- Deadlock handling is vital in lock-based systems.  
- **Snapshot isolation** balances performance and correctness in modern DBMS.


In [10]:
import threading
import time
from collections import defaultdict
from copy import deepcopy

# ===========================================
# ACID Transaction Simulation Components
# ===========================================

# Transaction States
ACTIVE = "ACTIVE"
WAITING = "WAITING"
COMMITTED = "COMMITTED"
ABORTED = "ABORTED"

# ===========================================
# Lock Manager (2PL) + Deadlock Detection
# ===========================================
class LockManager:
    def __init__(self):
        self.locks = {}  # {resource: (lock_type, transaction_id)}
        self.waiting = defaultdict(list)  # wait-for graph
        self.lock = threading.Lock()

    def acquire_lock(self, tid, resource, lock_type):
        with self.lock:
            if resource not in self.locks:
                self.locks[resource] = (lock_type, tid)
                print(f"[2PL] T{tid} acquired {lock_type} lock on {resource}")
                return True
            else:
                existing_type, existing_tid = self.locks[resource]
                if existing_tid == tid:
                    return True
                else:
                    # Conflict → wait
                    print(f"[2PL] T{tid} waiting for {resource} locked by T{existing_tid}")
                    self.waiting[tid].append(existing_tid)
                    return False

    def release_locks(self, tid):
        with self.lock:
            resources_to_free = [r for r, (_, holder) in self.locks.items() if holder == tid]
            for r in resources_to_free:
                del self.locks[r]
                print(f"[2PL] T{tid} released lock on {r}")
            if tid in self.waiting:
                del self.waiting[tid]

    def build_wait_for_graph(self):
        return dict(self.waiting)

class DeadlockDetector(threading.Thread):
    def __init__(self, lock_manager):
        super().__init__()
        self.lock_manager = lock_manager
        self.running = True

    def run(self):
        while self.running:
            time.sleep(1)
            graph = self.lock_manager.build_wait_for_graph()
            if self.detect_cycle(graph):
                print("\n*** DEADLOCK DETECTED! ***")
                self.resolve_deadlock(graph)

    def detect_cycle(self, graph):
        visited = set()
        def dfs(node, stack):
            if node in stack:
                return True
            stack.add(node)
            for neighbor in graph.get(node, []):
                if dfs(neighbor, stack):
                    return True
            stack.remove(node)
            return False
        return any(dfs(node, set()) for node in graph)

    def resolve_deadlock(self, graph):
        victim = max(graph.keys())
        print(f"[Deadlock] Aborting T{victim}")
        self.lock_manager.release_locks(victim)
        if victim in graph:
            del graph[victim]

# ===========================================
# MVCC with Snapshot Isolation
# ===========================================
class MVCCDatabase:
    def __init__(self):
        self.data = {}  # {key: [(version, value, tx_id, committed)]}
        self.lock = threading.Lock()

    def read(self, tid, key, snapshot_ts):
        with self.lock:
            if key not in self.data:
                return None
            # Find latest version <= snapshot_ts
            versions = sorted(self.data[key], key=lambda x: x[0], reverse=True)
            for version, value, tx_id, committed in versions:
                if version <= snapshot_ts and committed:
                    print(f"[MVCC] T{tid} reads {key}={value} (ver {version})")
                    return value
            return None

    def write(self, tid, key, value, ts):
        with self.lock:
            if key not in self.data:
                self.data[key] = []
            self.data[key].append((ts, value, tid, False))
            print(f"[MVCC] T{tid} writes {key}={value} (ver {ts})")

    def commit(self, tid):
        with self.lock:
            for key, versions in self.data.items():
                for i, (version, value, tx_id, committed) in enumerate(versions):
                    if tx_id == tid:
                        versions[i] = (version, value, tx_id, True)
            print(f"[MVCC] T{tid} committed all changes")

# ===========================================
# Transaction (supports both 2PL and MVCC)
# ===========================================
class Transaction(threading.Thread):
    def __init__(self, tid, mode, actions, lock_manager=None, mvcc_db=None, start_ts=None):
        super().__init__()
        self.tid = tid
        self.mode = mode  # '2PL' or 'MVCC'
        self.actions = actions
        self.lock_manager = lock_manager
        self.mvcc_db = mvcc_db
        self.start_ts = start_ts
        self.state = ACTIVE

    def run(self):
        print(f"T{self.tid} STARTED in {self.mode} mode")
        if self.mode == "2PL":
            self.run_2pl()
        else:
            self.run_mvcc()

    def run_2pl(self):
        for resource, op, value in self.actions:
            lock_type = "X" if op == "write" else "S"
            while not self.lock_manager.acquire_lock(self.tid, resource, lock_type):
                self.state = WAITING
                time.sleep(0.5)
            self.state = ACTIVE
            if op == "read":
                print(f"[2PL] T{self.tid} reads {resource}")
            elif op == "write":
                print(f"[2PL] T{self.tid} writes {resource}={value}")
            time.sleep(1)
        self.lock_manager.release_locks(self.tid)
        self.state = COMMITTED
        print(f"[2PL] T{self.tid} COMMITTED")

    def run_mvcc(self):
        for resource, op, value in self.actions:
            if op == "read":
                self.mvcc_db.read(self.tid, resource, self.start_ts)
            elif op == "write":
                self.mvcc_db.write(self.tid, resource, value, self.start_ts)
            time.sleep(1)
        self.mvcc_db.commit(self.tid)
        self.state = COMMITTED

# ===========================================
# MAIN SIMULATION
# ===========================================
if __name__ == "__main__":
    print("=== Week 5: Transaction Processing & Concurrency Control Simulation ===")

    # ---------------- 2PL with Deadlock ----------------
    print("\n--- 2PL with Deadlock Detection ---")
    lm = LockManager()
    dd = DeadlockDetector(lm)
    dd.start()

    t1 = Transaction(1, "2PL", [("A", "write", 100), ("B", "write", 200)], lock_manager=lm)
    t2 = Transaction(2, "2PL", [("B", "write", 300), ("A", "write", 400)], lock_manager=lm)
    t1.start()
    t2.start()
    t1.join()
    t2.join()
    dd.running = False
    dd.join()

    # ---------------- MVCC with Snapshot Isolation ----------------
    print("\n--- MVCC with Snapshot Isolation ---")
    mvcc_db = MVCCDatabase()

    # Initial data
    mvcc_db.write(0, "X", 10, 0)
    mvcc_db.commit(0)

    # T3 reads old value while T4 updates in parallel
    t3 = Transaction(3, "MVCC", [("X", "read", None), ("X", "read", None)],
                     mvcc_db=mvcc_db, start_ts=1)
    t4 = Transaction(4, "MVCC", [("X", "write", 50), ("X", "write", 60)],
                     mvcc_db=mvcc_db, start_ts=2)
    t3.start()
    t4.start()
    t3.join()
    t4.join()

    print("\nSimulation complete.")


=== Week 5: Transaction Processing & Concurrency Control Simulation ===

--- 2PL with Deadlock Detection ---
T1 STARTED in 2PL mode
[2PL] T1 acquired X lock on A
[2PL] T1 writes A=100
T2 STARTED in 2PL mode
[2PL] T2 acquired X lock on B
[2PL] T2 writes B=300
[2PL] T1 waiting for B locked by T2
[2PL] T2 waiting for A locked by T1
[2PL] T1 waiting for B locked by T2
[2PL] T2 waiting for A locked by T1

*** DEADLOCK DETECTED! ***
[Deadlock] Aborting T2
[2PL] T2 released lock on B
[2PL] T1 acquired X lock on B
[2PL] T1 writes B=200
[2PL] T2 waiting for A locked by T1
[2PL] T2 waiting for A locked by T1

*** DEADLOCK DETECTED! ***
[Deadlock] Aborting T2
[2PL] T1 released lock on A
[2PL] T1 released lock on B
[2PL] T1 COMMITTED
[2PL] T2 acquired X lock on A
[2PL] T2 writes A=400
[2PL] T2 released lock on A
[2PL] T2 COMMITTED

--- MVCC with Snapshot Isolation ---
[MVCC] T0 writes X=10 (ver 0)
[MVCC] T0 committed all changes
T3 STARTED in MVCC mode
[MVCC] T3 reads X=10 (ver 0)
T4 STARTED in MV

This run shows the concepts from **Week 5** in action:  

## 2PL with Deadlock Detection  
1. **T1 locks A**, **T2 locks B**, then both try to lock the other’s resource → **circular wait**.  
2. **Deadlock detector** finds a cycle in the wait-for graph.  
3. **T2 is aborted** as the victim, releasing its locks so **T1** can finish.  
4. After **T1** commits, **T2 restarts** and completes successfully.  

**Takeaway:** Demonstrates **serializability** but also the **blocking** and **restart overhead** of lock-based systems.  

---

## MVCC with Snapshot Isolation  
1. **T3** starts with a **snapshot** of committed data where $X = 10$.  
2. **T4** writes new versions of $X$ ($50$, then $60$), but **T3** still sees the old value ($10$) because of snapshot isolation.  
3. **No blocking occurs** — both transactions commit without conflict.  

**Takeaway:** Shows **MVCC’s non-blocking reads** and **isolation through versioning**.
